In [1]:
' today, we are handling calculators and code interpreters by building a tool that lets the'
'LLM execute python math'

# environment setup and imports
import os
from dotenv import load_dotenv, find_dotenv
from llama_index.core import StorageContext, VectorStoreIndex, Settings
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from llama_index.core.tools import QueryEngineTool, ToolMetadata, FunctionTool
from llama_index.core.agent import ReActAgent
from llama_index.llms.groq import Groq
from pymongo import MongoClient, AsyncMongoClient
from llama_index.tools.tavily_research import TavilyToolSpec
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


load_dotenv(find_dotenv())

# Setting up llm and embedding model
llm = Groq(
    model = 'llama-3.3-70B-versatile',
    api_key= os.getenv('GROQ_API_KEY')
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name= 'BAAI/bge-m3'
)

Settings.llm = llm

print('Environment Setup complete 😁')

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment Setup complete 😁


In [9]:
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)

print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [ ]:
# here we are going to add the Financial Logic functions 
# this cell defines the raw python logic for calculations to ensure the agent doesn't hallucinate math.

def calculate_cagr(beginning_value: float, ending_value: float, periods: int) -> float:
    """Calculates the Compound Annual Growth Rate (CAGR). """
    if beginning_value <= 0 or periods <= 0:
        return 0.0
    cagr = (ending_value / beginning_value) ** (1 / periods) -1
    return round(cagr * 100, 2)

def calculate_net_margin(net_income: float, revenue: float) -> float:
    """Calculates the Net Profit Margin as a percentage."""
    if revenue <= 0:
        return 0.0
    margin = (net_income / revenue) * 100
    return round(margin, 2)

def calculate_yoy_growth(current_val: float, previous_val: float) -> float:
    """Calculates Year-Over-Year (YoY) growth percentage """ 
    if previous_val == 0:
        return 0.0
    growth = ((current_val - previous_val) / previous_val) * 100
    return round(growth, 2)

print("🛣️ Financial functions defined.")

🛣️ Financial functions defined.


In [11]:
# now we are going to convert functions into Agent tools
# this cell Wraps python functions into FunctionTool objects so the Agent can "see" them

cagr_tool = FunctionTool.from_defaults(
    fn = calculate_cagr,
    name= "cagr_calculator",
    description= "Calculates CAGR percentage. Requirements: beginning_value (float), ending_value (float), periods (int)."
)

marginTool = FunctionTool.from_defaults(
    fn = calculate_net_margin,
    name= "net_margin_calculator",
    description= "Calculates Net Profit Margin percentage. Requirements: net_income (float), revenue (float)."
)

yoyTool = FunctionTool.from_defaults(
    fn = calculate_yoy_growth,
    name = "yoy_growth_calculator",
    description= "Calculates YoY growth percentage. Requirements: current_val (float), previous_val (float)."
)

# adding tavily search tool as well if needed
searchTool = TavilyToolSpec(api_key= os.getenv('TAVILY_API_KEY')).to_tool_list()[0]

print('👍 Tools initialized.')



👍 Tools initialized.


In [12]:
# in this cell, we are going to initialize the  ReAct agent
# here we configure the reasoning engine and the system prompt

# assembling all tools
tools = [apple_10k_expert, cagr_tool, marginTool, yoyTool, searchTool]

# system prompt to force the agent to use the tools for math
systemPrompt = """ 
You are a Senior Apple Financial Analyst. Your goal is to provide accurate financial insights.
1. Use 'apple_10k_expert' to find specific dollar amounts from the reports.
2. For ANY calculation (growth, margins, CAGR), you MUST use the appropriate calculator tool.
3. Never calculate percentages or multi-year growth manualy in your head.
4. If a number is in millions (e.g., $383,285), ensure you use the full numeric value for the calculators.
5. If the user asks for specific years (e.g., 2021 and 2023), do not substitute them with intermediate years like 2022 in your final calculation.
5. If the user asks for specific years (e.g., 2022 and 2023), do not substitute them with intermediate years like 2022 in your final calculation.
6. Before calling a calculator tool, you MUST explicitly state the Year and Value you found. If the user asks for 2021 and you only find 2022, you must report that the 2021 data is missing rather than substituting it.
"""

agent = ReActAgent(
    tools= tools,
    llm = llm,
    verbose= True,
    system_prompt= systemPrompt
)

print('🤝 Agent is ready for financial analysis.')

🤝 Agent is ready for financial analysis.


In [13]:
# the 'CFO Test' Execution
# we use this cell to run the complex query that triggers the full "Retrieve then Calculate" loop

# testCase: Multi-step retrieval and Calculation
query = "What was Apple's total net sales in 2021 and 2023? Use those figures to calculate the 2-year CAGR."

response = await agent.run(user_msg= query)

print("\n------FINAL RESPONSE------")
print(response)

[tick] add: AgentWorkflowStartEvent(user_msg="What was Apple's total net sales in 2021 and 2023?...", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What was Apple's total net sales in 2021 and 2023? Use those figures ...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text=" \nYou are a Senior Apple Financial Analyst. Your goal is to prov...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kh8ddxqef7psaxah8yq2j3xc` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98808, Requested 1815. Please try again in 8m58.272s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [7]:
# the 'CFO Test' Execution
# we use this cell to run the complex query that triggers the full "Retrieve then Calculate" loop

# testCase: Multi-step retrieval and Calculation
query = "Retrieve the 'Consolidated Statements of Operations' for 2021 and 2023. Provide the exact million-dollar figures and calculate the 2-year CAGR."

response = await agent.run(user_msg= query)

print("\n------FINAL RESPONSE------")
print(response)

[tick] add: AgentWorkflowStartEvent(user_msg="Retrieve the 'Consolidated Statements of Operation...", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="Retrieve the 'Consolidated Statements of Operations' for 2021 and 202...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text=" \nYou are a Senior Apple Financial Analyst. Your goal is to prov...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_

In [8]:
# additional otional diagnostic Tool inspection
for tool in tools:
    print(f"Tool Name: {tool.metadata.name}")
    print(f"Description: {tool.metadata.description}")
    print(f"Parameters: {tool.metadata.get_parameters_dict()}\n")

Tool Name: apple_10k_expert
Description: Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors.
Parameters: {'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'type': 'object'}

Tool Name: cagr_calculator
Description: Calculates CAGR percentage. Requirements: beginning_value (float), ending_value (float), periods (int).
Parameters: {'properties': {'beginning_value': {'title': 'Beginning Value', 'type': 'number'}, 'ending_value': {'title': 'Ending Value', 'type': 'number'}, 'periods': {'title': 'Periods', 'type': 'integer'}}, 'required': ['beginning_value', 'ending_value', 'periods'], 'type': 'object'}

Tool Name: net_margin_calculator
Description: Calculates Net Profit Margin percentage. Requirements: net_income (float), revenue (float).
Parameters: {'properties': {'net_income': {'title': 'Net Income', 'type': 'number'}, 'revenue': {'title': 'Revenue', 'type': 'number'}}, 'required': ['net_income', 'reve